In [ ]:
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
import re
from collections import Counter
from itertools import groupby, pairwise

import matplotlib.pyplot as plt
import pandas as pd

from drzavni_posao.transcription import MlflowTracker, Stage

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.transcription.character_assignment_eda")

In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
RUN_ID = "<character-assignment-run-id>"
TOP_N = 20

assert not RUN_ID.startswith("<"), "Set RUN_ID to a completed assignment run"
tracker = MlflowTracker(EXPERIMENT_NAME)

In [ ]:
TOKEN_RE = re.compile(r"[^\W\d_]+", re.UNICODE)
STOPWORDS = {
    "a",
    "ako",
    "ali",
    "bi",
    "bih",
    "bila",
    "bili",
    "bio",
    "da",
    "do",
    "ga",
    "i",
    "ili",
    "ima",
    "iz",
    "ja",
    "je",
    "kad",
    "kao",
    "ko",
    "koji",
    "mi",
    "na",
    "ne",
    "nije",
    "ništa",
    "od",
    "on",
    "ona",
    "pa",
    "po",
    "sam",
    "se",
    "si",
    "smo",
    "su",
    "šta",
    "što",
    "ta",
    "ti",
    "to",
    "u",
    "za",
}


def normalize_word(text: str) -> str:
    return "".join(TOKEN_RE.findall(text.lower()))


def phrases(tokens: list[str], size: int):
    return (" ".join(tokens[i : i + size]) for i in range(len(tokens) - size + 1))

In [ ]:
states = tracker.episode_run_states(RUN_ID)
incomplete = sorted(
    key
    for key, state in states.items()
    if Stage.CHARACTER_ASSIGNER not in state.completed_stages
)
assert states, f"Run {RUN_ID} has no episode runs"
assert not incomplete, f"Character assignment is incomplete for: {incomplete}"

segment_rows = []
word_rows = []
for episode_key, state in sorted(
    states.items(),
    key=lambda item: int(item[0]) if item[0].isdigit() else item[0],
):
    transcript = tracker.stage_transcript(state.run_id, Stage.CHARACTER_ASSIGNER)
    for segment_index, segment in enumerate(transcript.segments):
        character = segment.get("speaker") or "[unassigned]"
        start = segment.get("start")
        end = segment.get("end")
        segment_duration = (
            max(0.0, end - start)
            if start is not None and end is not None
            else 0.0
        )
        tokens = []
        timed_word_seconds = 0.0
        scores = []
        for word_index, word in enumerate(segment.get("words", [])):
            token = normalize_word(word.get("word", ""))
            if token:
                tokens.append(token)
            word_start = word.get("start")
            word_end = word.get("end")
            word_duration = (
                max(0.0, word_end - word_start)
                if word_start is not None and word_end is not None
                else 0.0
            )
            timed_word_seconds += word_duration
            score = word.get("score")
            if score is not None:
                scores.append(score)
            word_rows.append(
                {
                    "episode_key": episode_key,
                    "segment_index": segment_index,
                    "word_index": word_index,
                    "character": character,
                    "word": word.get("word", ""),
                    "token": token,
                    "start": word_start,
                    "end": word_end,
                    "seconds": word_duration,
                    "score": score,
                }
            )

        segment_rows.append(
            {
                "episode_key": episode_key,
                "segment_index": segment_index,
                "character": character,
                "start": start,
                "end": end,
                "seconds": segment_duration,
                "word_seconds": timed_word_seconds,
                "word_count": len(tokens),
                "mean_word_score": sum(scores) / len(scores) if scores else None,
                "text": segment.get("text", ""),
                "tokens": tokens,
            }
        )

segments = pd.DataFrame(segment_rows)
words = pd.DataFrame(word_rows)
logger.info(
    "Loaded %d episodes, %d segments, and %d words",
    len(states),
    len(segments),
    len(words),
)
segments.head()

In [ ]:
segments = segments.sort_values(["episode_key", "start"]).reset_index(drop=True)
new_turn = (
    segments["episode_key"].ne(segments["episode_key"].shift())
    | segments["character"].ne(segments["character"].shift())
)
segments["turn_id"] = new_turn.cumsum()
turns = (
    segments.groupby(["episode_key", "turn_id", "character"], as_index=False)
    .agg(
        start=("start", "min"),
        end=("end", "max"),
        segment_count=("segment_index", "size"),
        word_count=("word_count", "sum"),
        speech_seconds=("seconds", "sum"),
    )
)
turns["span_seconds"] = turns["end"] - turns["start"]

spoken_words = words[words["token"].ne("")].copy()
summary = pd.Series(
    {
        "episodes": segments["episode_key"].nunique(),
        "characters": segments["character"].nunique(),
        "segments": len(segments),
        "speaker_turns": len(turns),
        "words": len(spoken_words),
        "unique_words": spoken_words["token"].nunique(),
        "segment_speech_hours": segments["seconds"].sum() / 3600,
        "aligned_word_hours": words["seconds"].sum() / 3600,
        "mean_words_per_segment": segments["word_count"].mean(),
        "mean_words_per_turn": turns["word_count"].mean(),
    },
    name="value",
)
summary

In [ ]:
character_segments = segments.groupby("character").agg(
    episodes=("episode_key", "nunique"),
    segments=("segment_index", "size"),
    speech_seconds=("seconds", "sum"),
    aligned_word_seconds=("word_seconds", "sum"),
)
character_words = spoken_words.groupby("character").agg(
    words=("token", "size"),
    unique_words=("token", "nunique"),
    mean_alignment_score=("score", "mean"),
)
character_turns = turns.groupby("character").agg(
    turns=("turn_id", "size"),
    mean_turn_words=("word_count", "mean"),
    median_turn_words=("word_count", "median"),
    longest_turn_words=("word_count", "max"),
)

character_summary = character_segments.join(
    [character_words, character_turns],
    how="outer",
).fillna(0)
character_summary["speech_minutes"] = character_summary["speech_seconds"] / 60
character_summary["words_per_minute"] = (
    60 * character_summary["words"] / character_summary["speech_seconds"]
)
character_summary["vocabulary_ratio"] = (
    character_summary["unique_words"] / character_summary["words"]
)
character_summary["word_share"] = character_summary["words"] / character_summary[
    "words"
].sum()
character_summary["speech_share"] = character_summary["speech_seconds"] / (
    character_summary["speech_seconds"].sum()
)
character_summary = character_summary.sort_values("words", ascending=False)

display(
    character_summary[
        [
            "episodes",
            "words",
            "unique_words",
            "speech_minutes",
            "words_per_minute",
            "turns",
            "mean_turn_words",
            "longest_turn_words",
            "word_share",
            "speech_share",
            "mean_alignment_score",
        ]
    ].round(3)
)

In [ ]:
episode_summary = segments.groupby("episode_key").agg(
    segments=("segment_index", "size"),
    characters=("character", "nunique"),
    words=("word_count", "sum"),
    speech_seconds=("seconds", "sum"),
    aligned_word_seconds=("word_seconds", "sum"),
    first_speech=("start", "min"),
    last_speech=("end", "max"),
)
episode_summary["speech_minutes"] = episode_summary["speech_seconds"] / 60
episode_summary["words_per_minute"] = (
    60 * episode_summary["words"] / episode_summary["speech_seconds"]
)
episode_summary["speech_coverage"] = episode_summary["speech_seconds"] / (
    episode_summary["last_speech"] - episode_summary["first_speech"]
)

episode_character_words = segments.groupby(
    ["episode_key", "character"]
)["word_count"].sum()
dominant_character = episode_character_words.groupby(level=0).idxmax().map(
    lambda key: key[1]
)
episode_summary["dominant_character"] = dominant_character

episode_summary = episode_summary.sort_index(
    key=lambda index: index.map(lambda key: int(key) if str(key).isdigit() else key)
)
display(episode_summary.round(3))
episode_summary.describe().round(3)

In [ ]:
def counter_frame(counter: Counter, label: str, limit: int = TOP_N) -> pd.DataFrame:
    return pd.DataFrame(counter.most_common(limit), columns=[label, "count"])


word_counts = Counter(spoken_words["token"])
content_word_counts = Counter(
    token for token in spoken_words["token"] if token not in STOPWORDS
)
bigram_counts = Counter()
trigram_counts = Counter()
for tokens in segments["tokens"]:
    bigram_counts.update(phrases(tokens, 2))
    trigram_counts.update(phrases(tokens, 3))

print("Most common words")
display(counter_frame(word_counts, "word"))
print("Most common content words")
display(counter_frame(content_word_counts, "word"))
print("Most common two-word phrases")
display(counter_frame(bigram_counts, "phrase"))
print("Most common three-word phrases")
display(counter_frame(trigram_counts, "phrase"))

In [ ]:
CHARACTER = character_summary.index[0]

character_tokens = spoken_words.loc[
    spoken_words["character"].eq(CHARACTER), "token"
]
character_word_counts = Counter(character_tokens)
character_bigrams = Counter()
character_trigrams = Counter()
for tokens in segments.loc[segments["character"].eq(CHARACTER), "tokens"]:
    character_bigrams.update(phrases(tokens, 2))
    character_trigrams.update(phrases(tokens, 3))

print(f"Lexical profile: {CHARACTER}")
display(counter_frame(character_word_counts, "word"))
display(counter_frame(character_bigrams, "phrase"))
display(counter_frame(character_trigrams, "phrase"))

global_total = sum(word_counts.values())
character_total = sum(character_word_counts.values())
distinctive = []
for token, count in character_word_counts.items():
    if count < 5 or token in STOPWORDS:
        continue
    global_count = word_counts[token]
    lift = (count / character_total) / (global_count / global_total)
    distinctive.append(
        {
            "word": token,
            "character_count": count,
            "global_count": global_count,
            "frequency_lift": lift,
        }
    )

display(
    pd.DataFrame(distinctive)
    .sort_values(["frequency_lift", "character_count"], ascending=False)
    .head(TOP_N)
    .round(2)
)

In [ ]:
print("Longest speaker turns by word count")
display(
    turns.nlargest(TOP_N, "word_count")[
        [
            "episode_key",
            "character",
            "start",
            "end",
            "word_count",
            "speech_seconds",
            "segment_count",
        ]
    ].round(3)
)

print("Longest individual segments")
display(
    segments.nlargest(TOP_N, "word_count")[
        [
            "episode_key",
            "character",
            "start",
            "end",
            "word_count",
            "seconds",
            "text",
        ]
    ].round(3)
)

print("Lowest-confidence aligned words")
display(
    words.dropna(subset=["score"])
    .nsmallest(TOP_N, "score")[
        ["episode_key", "character", "word", "start", "end", "score"]
    ]
    .round(3)
)

In [ ]:
transitions = Counter()
for _, episode in segments.groupby("episode_key", sort=False):
    speakers = [
        speaker
        for speaker, _ in groupby(episode.sort_values("start")["character"])
    ]
    transitions.update(pairwise(speakers))

transition_rows = [
    {"from": source, "to": target, "count": count}
    for (source, target), count in transitions.most_common(TOP_N)
]
print("Most frequent character handoffs")
display(pd.DataFrame(transition_rows))

cooccurrences = Counter()
for _, episode in segments.groupby("episode_key"):
    characters = sorted(set(episode["character"]))
    cooccurrences.update(
        (characters[i], characters[j])
        for i in range(len(characters))
        for j in range(i + 1, len(characters))
    )

cooccurrence_rows = [
    {"character_a": first, "character_b": second, "episodes": count}
    for (first, second), count in cooccurrences.most_common(TOP_N)
]
print("Character pairs appearing together most often")
display(pd.DataFrame(cooccurrence_rows))

In [ ]:
top_characters = character_summary.head(15).sort_values("words")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top_characters["words"].plot.barh(ax=axes[0], title="Words spoken by character")
axes[0].set_xlabel("Words")
axes[0].set_ylabel("Character")
top_characters["speech_minutes"].plot.barh(
    ax=axes[1],
    title="Segment speech duration by character",
)
axes[1].set_xlabel("Minutes")
axes[1].set_ylabel("Character")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(episode_summary["speech_minutes"], episode_summary["words"], alpha=0.7)
ax.set_title("Episode words versus speech duration")
ax.set_xlabel("Segment speech duration (minutes)")
ax.set_ylabel("Words")
plt.show()